# Pancreas and pancreatic lymph node: tissue-specific protein profiles

Compare sample coverage, cell identities and protein measurements across pancreas and pancreatic lymph node (pLN). Each tissue has its own native typing system, normalization and UMAP. The pancreas carries final hierarchy assignments; the dedicated pLN results are provisional rule candidates. The pLN views keep strict candidates and show the broad protein evidence recovered for previously unmatched cells. Section 2 audits the high fraction that failed every broad lineage rule across all 19 acquisitions.

The notebooks require the November 13 Islet Explorer protein release that excludes EpCAM from typing and rebuilds the pancreas UMAP without that marker. The load step checks the release ID and the marker list saved with the embedding.

Adapted from the [original PhenoCycler analysis notebook](https://github.com/smith6jt-cop/Phenocycler_Analysis/blob/ee170f13ebb892bc395bb289625df038ad0fadb1/notebooks/05_tissue_comparisons.ipynb).

## 1. Load the tissue-specific samples

The dataset includes all available pancreas donors and only the subset with pLN tissue. Select a donor with both tissues and a marker they share. The dedicated pLN candidate hierarchy also omits EpCAM. Its UMAP was refit without EpCAM using the revised candidate-stratified sample. Its marker calibration stays tissue specific.

The default `PLN_METHOD = "review"` retains the strict subtype candidates, including ambiguous and unavailable outcomes. For the unmatched cells, it shows a broad immune review label only when the historical cluster and the cell's own marker group agree. The default separates eligible macrophages into two subtypes using their own galectin-3 measurement. CD68 or CD163 positivity supports the parent, including double-positive cells. Previously resolved conflicting lineages retain their tissue-specific calls. Remaining unresolved cells retain their review status. Use `"strict"` to see the original hard-gate calls or `"scimap"` for the continuous-score sensitivity. All three views use the same protein measurements, coordinates and admitted-cell denominator. `PLN_PROFILE_SCOPE` selects donor, disease or cohort for the pLN protein panel.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm
import anndata as ad
import scanpy as sc
from IPython.display import display

# From a separate checkout, set PHENOCYCLER_DATA to the folder containing the .h5ad files.
cwd = Path.cwd()
candidates = [Path(os.environ["PHENOCYCLER_DATA"])] if "PHENOCYCLER_DATA" in os.environ else [
    cwd / "analysis" / "phenocycler" / "scanpy", cwd / "phenocycler" / "scanpy",
    cwd / "scanpy", cwd.parent / "scanpy", cwd]
data = next((d for d in candidates if (d / "typing_release.json").is_file()), None)
if data is None:
    raise FileNotFoundError("Set PHENOCYCLER_DATA to the no-EpCAM Scanpy release folder containing typing_release.json.")
sc.set_figure_params(dpi=100, facecolor="white", fontsize=10, frameon=False)
plt.rcParams.update({"axes.spines.top":False, "axes.spines.right":False})

# Locate the shared plotting module when running from a notebook, its parent, or the Senior checkout.
import sys
helper_paths = [cwd, cwd/"notebooks", cwd/"phenocycler"/"notebooks", cwd/"analysis"/"phenocycler"/"notebooks",
                data.parent/"notebooks"]
helper_dir = next((p for p in helper_paths if (p/"phenotype_signatures.py").is_file()), None)
if helper_dir is None:
    raise FileNotFoundError("Run from the Senior checkout or its analysis/phenocycler/notebooks directory.")
sys.path.insert(0, str(helper_dir))
from phenotype_signatures import (PAN_STRUCTURE, HORMONES, IMMUNE, T_CELL, DISPLAY_NAMES,
                                  donor_heatmap, show_profiles, conflict_summary, validate_release)
from phenotype_signatures import (hierarchy_tables, cell_subtypes, phenotype_colors, phenotype_key, names_in_order, subtype_display_name)
release = validate_release(data)

DONOR = "115"
MARKER = "CD3e"
PLN_METHOD = "review"  # "review", "strict", or "scimap"
PLN_PROFILE_SCOPE = "donor"  # "donor", "disease" or "cohort"
if PLN_METHOD not in {"review", "scimap", "strict"} or PLN_PROFILE_SCOPE not in {"donor","disease","cohort"}:
    raise ValueError("Choose one of the listed pLN methods and scopes")
method_name = {"review":"Strict + broad review", "scimap":"SCIMAP sensitivity", "strict":"Strict rules"}[PLN_METHOD]
pan = sc.read_h5ad(data / "pancreas_umap.h5ad")
pln = sc.read_h5ad(data / "pln_umap.h5ad")
counts = pd.read_parquet(data / "donor_counts.parquet")
counts["donor"] = counts.donor.astype(str)
evidence = pd.read_parquet(data / "donor_markers.parquet")
evidence["donor"] = evidence.donor.astype(str)
counts, evidence = hierarchy_tables(counts, evidence)
# The two candidate methods retain separate count and marker tables.
score_counts = pd.read_parquet(data / "pln_scimap_candidate_proportions.parquet")
score_evidence = pd.read_parquet(data / "pln_scimap_candidate_markers.parquet")
for table in [score_counts,score_evidence]:
    table.rename(columns={"scimap_candidate_detail":"label"}, inplace=True)
    table["level"] = "subtype"
    table["donor"] = table.donor.astype(str)
score_counts["comparable"] = score_counts.assessable
score_counts["parent_cells"] = score_counts.root_cells
if score_counts.duplicated(["donor","label"]).any() or score_evidence.duplicated(["donor","label","marker"]).any():
    raise ValueError("SCIMAP donor and subtype rows must be unique before plotting")
pln_counts = score_counts if PLN_METHOD == "scimap" else counts.loc[counts.tissue.eq("pLN")]
pln_evidence = score_evidence if PLN_METHOD == "scimap" else evidence.loc[evidence.tissue.eq("pLN")]

review_counts = pd.read_parquet(data / "pln_unmatched_review_counts.parquet")
review = pd.read_parquet(data / "pln_unmatched_review_markers.parquet")
if PLN_METHOD == "review":
    pln_counts = pd.read_parquet(data / "pln_macrophage_counts.parquet")
    pln_evidence = pd.read_parquet(data / "pln_macrophage_markers.parquet")
    assignments = pd.read_parquet(data / "pln_macrophage_embedding.parquet").set_index("object_id",verify_integrity=True)
    assigned = assignments.reindex(pln.obs_names)
    if assigned.pln_macrophage_detail.isna().any() or not np.array_equal(assigned.donor.astype(str), pln.obs.donor.astype(str)):
        raise ValueError("Macrophage review and embedding have different islet explorer cell IDs or donors")
    pln.obs["pln_macrophage_detail"] = pd.Categorical(assigned.pln_macrophage_detail)
    assert pln_counts.groupby("donor",observed=True).cells.sum().eq(
        pln_counts.groupby("donor",observed=True).eligible_cells.first()).all()
n = counts.loc[counts.level.eq("primary")].groupby(["tissue","donor"],observed=True).eligible_cells.first().unstack("tissue")
paired = n.dropna().index
if DONOR not in paired: raise ValueError(f"Choose a donor with both tissues: {sorted(paired,key=int)}")
if MARKER not in set(pan.var_names) & set(pln.var_names): raise ValueError("Choose a shared measured marker.")
print(f"{n.Pancreas.notna().sum()} donors with pancreas · {n.pLN.notna().sum()} with pLN · {len(paired)} paired donors")
display(n.loc[[DONOR]].astype(int).rename(columns={"Pancreas":"Admitted pancreas cells","pLN":"Admitted pLN cells"}))

## 2. Why did lymph-node cells fail the broad lineage rules?

All 19 dedicated sections used a candidate workflow based on whole-cell QuPath mean fluorescence. Each section and marker had its own background floor and upper intensity bound. The assay counted a marker as positive at 10% of that calibrated range. A cell could express a lineage marker and still fail the parent rule if a required negative marker was positive.

The bars partition every unmatched cell into three reasons and give the percentage of **all admitted cells**. The table below them reports which positive signatures were blocked by exclusion markers. A cell can appear under several roots in that table. Full-cell replay verifies that no cell reported as unmatched actually passed a complete root. The biological identity of these cells remains unresolved.

The October within-section cluster analysis provides a second reading of the same native cell IDs. Its cell-level marker check measures agreement with the cluster label. These two analyses share the underlying protein measurements and have different assignment criteria.

In [ ]:
audit = pd.read_parquet(data / "pln_no_match_by_donor.parquet")
root_audit = pd.read_parquet(data / "pln_no_match_by_root.parquet")
channel_audit = pd.read_parquet(data / "pln_no_match_marker_qc.parquet")
for table in [audit, root_audit, channel_audit]:
    table["donor"] = table.donor.astype(str)
reasons = {"no_root_positive":"No broad-lineage positive marker",
           "partial_positive_only":"Partial positive signature",
           "complete_positive_blocked":"Positive signature blocked by a negative gate"}
audit = audit.sort_values("no_match_pct").reset_index(drop=True)
assert audit[list(reasons)].sum(axis=1).eq(audit.no_match).all()
census = counts.query("tissue == 'pLN' and level == 'primary' and label == 'No rule match'").set_index("donor")
assert audit.no_match.eq(audit.donor.map(census.cells)).all()
assert audit.admitted.eq(audit.donor.map(census.eligible_cells)).all()
assert root_audit.complete_positive_no_veto.eq(0).all()
fig, ax = plt.subplots(figsize=(9,6.4))
left = np.zeros(len(audit))
for (field,label),color in zip(reasons.items(), ["#69829a","#d8a45e","#9c4f5f"]):
    width = 100*audit[field]/audit.admitted
    ax.barh(audit.donor, width, left=left, height=.72, color=color, label=label)
    left += width.to_numpy()
ax.set(xlabel="Admitted cells in the section (%)", ylabel="Donor", xlim=(0, 58),
       title="Dedicated pLN · cells with no matching lineage rule")
for i,pct in enumerate(left):
    ax.text(pct+.5,i,f"{pct:.1f}%",va="center",fontsize=8)
ax.legend(loc="lower center",bbox_to_anchor=(.5,-.26),ncol=2,fontsize=8,frameon=False)
fig.subplots_adjust(left=.11,right=.98,top=.93,bottom=.22)
plt.show()
display(audit.set_index("donor"))
display(root_audit.groupby(["root","required_positive","any_positive","required_negative"],observed=True)
        [["complete_positive_blocked","complete_positive_no_veto"]].sum())
channels = channel_audit.groupby("marker",observed=True)[["positive","measured"]].sum()
channels["positive_pct_in_unmatched"] = 100*channels.positive/channels.measured
display(channels.sort_values("positive_pct_in_unmatched",ascending=False).round(2))

prior = pd.read_parquet(data / "pln_no_match_prior_clusters.parquet")
assert prior.groupby("donor",observed=True).cells.sum().sort_index().equals(audit.set_index("donor").no_match.sort_index())
cluster_check = prior.groupby("cluster_class",observed=True)[["cells","named_immune","cell_agrees_with_named_cluster","evaluated_for_named_cluster_agreement"]].sum()
cluster_check["agreement_pct"] = 100*cluster_check.cell_agrees_with_named_cluster / cluster_check.evaluated_for_named_cluster_agreement.replace(0,np.nan)
display(cluster_check.round(1))

### Protein evidence in the unmatched cells

Four broad immune populations can be reviewed by joining cells to the earlier within-donor protein clusters by native cell ID. A broad label is shown when the cell's strongest immune marker group agrees with its named cluster. The remaining rows retain the reason that a broad label could not be supported. The UMAP displays the selected donor's sampled cells. The protein summary uses **all unmatched cells** in that donor. Dot color is min–max scaled within each marker column, and dot area is the fraction of measured cells at or above the section's positivity threshold.

The cell-level file `pln_lineage_review_cells.parquet` contains the native ID, strict candidate outcome, broad review label, cell-level marker agreement and the independent own-compartment gate result. The same fields are in `pln_umap.h5ad` for its display sample.

Among the 2,064,910 cells still unresolved after this review, 266,412 passed an immune parent marker and the pan-cytokeratin exclusion. Another 497,197 lacked all five parent markers but were positive for CD79a, CD163, Iba1, MPO or CD66. The residual-evidence table gives these counts by donor. Its paired-marker column flags CD3e with CD4 or CD8, CD20 with CD79a, at least two of CD68, Iba1, CD163 and CD14, or MPO with CD66. These patterns identify cells for further review.

In [ ]:
from phenotype_signatures import draw_matrix, full_panel

donor_sample = pln[pln.obs.donor.astype(str).eq(DONOR) & pln.obs.candidate_state.eq("no_match")
                   & np.isfinite(pln.obsm["X_umap"]).all(axis=1)].copy()
if donor_sample.n_obs:
    donor_sample.obs["Broad protein evidence"] = donor_sample.obs.pln_lineage_review.astype("string").fillna(
        donor_sample.obs.pln_lineage_review_status.astype("string")).astype("category")
    sc.pl.umap(donor_sample, color="Broad protein evidence", size=35, frameon=False,
               title=f"Donor {DONOR} · unmatched cells on the pLN UMAP ({donor_sample.n_obs:,} mapped)")
select = review.donor.astype(str).eq(DONOR)
donor_review = review.loc[select].rename(columns={"mean_scaled":"mean","positive_fraction":"fraction",
                                                  "review_group":"label"}).set_index(["label","marker"])
support = review_counts[review_counts.donor.astype(str).eq(DONOR)].groupby("review_group",observed=True).cells.sum()
assert support.sum() == audit.set_index("donor").at[DONOR,"no_match"]
groups = support.sort_values(ascending=False).index.to_list()
panel = full_panel(evidence, "pLN")
draw_matrix(donor_review, groups, panel, [f"{g}  ·  {int(support[g]):,} cells" for g in groups],
            measure="dotplot", tissue="pLN", title=f"Donor {DONOR} · marker evidence within unmatched pLN cells",
            scope_note=f"{len(panel)} proteins  |  Gray × = no reportable measurement")
display(donor_review.reset_index()[["label","marker","n_measured","n_positive","mean","fraction"]])
residual = pd.read_parquet(data / "pln_residual_marker_support.parquet").set_index("donor")
unresolved_counts = review_counts.loc[review_counts.review_status.ne("Cell and named cluster agree")].groupby("donor", observed=True).cells.sum()
pd.testing.assert_series_equal(residual.residual_cells.sort_index(), unresolved_counts.sort_index(),
                               check_names=False, check_index_type=False, check_dtype=False)
columns = {"residual_cells":"Unresolved cells",
           "immune_root_anchor_vetoed_panck":"Immune anchor and pan-cytokeratin",
           "complementary_without_root_anchor":"Complementary marker only (no immune anchor)",
           "any_paired_lineage_signature":"Paired lineage markers"}
residual_summary = residual[list(columns)].rename(columns=columns)
display(residual_summary.loc[[DONOR]].rename_axis("Donor"))
display(residual_summary.sum().rename("All 19 donors").to_frame().T)


### Sensitivity to cellular compartment and scoring method

The compartment check refits the same section-local gate separately on the raw whole-cell mean and on the registered membrane or cytoplasmic mean for each marker. An unavailable compartment measurement cannot satisfy a required negative. The table compares both with the released shifted whole-cell result.

The SCIMAP check applies the hierarchy with continuous protein scores, including the five broad lineage roots and the descendants assessable on the dedicated panel. For each section and marker, the background floor, original cutoff and upper bound map to 0, 0.5 and 1. The result shows how assigning the highest qualifying score changes candidate assignments. The retained parent and evidence-status fields show where a finer subtype could be supported. Original hand gates from a different acquisition do not establish the identity of these dedicated pLN cells.

SCIMAP can select a neural candidate from weak positive-marker evidence because required-negative signals also contribute to its score. Of the 618,744 former no-matches that it calls neural, 1,724 (0.28%) cross the B3TUBB positivity gate. Use these candidates to inspect sensitivity to the scoring rule. The review labels and unresolved outcomes are available on the same cell IDs.

In [ ]:
compartments = pd.read_parquet(data / "pln_compartment_roots.parquet")
root_changes = compartments[compartments.population.eq("source_unmatched")].groupby("root",observed=True)
display(root_changes[["released","cell_raw","own_raw"]].sum().rename(columns={
    "released":"Released whole cell","cell_raw":"Raw whole cell","own_raw":"Raw own compartment"}))
scored = pd.read_parquet(data / "pln_scimap_candidate_counts.parquet")
assert scored.cells.sum() == audit.admitted.sum()
transitions = scored.pivot_table(index="source_root",columns="scimap_candidate_root",values="cells",
                                 aggfunc="sum",fill_value=0,observed=True)
display(transitions)
unmatched = scored.loc[scored.candidate_state.eq("no_match")]
display(unmatched.pivot_table(index="scimap_candidate_root",columns="pln_lineage_review",values="cells",
                              aggfunc="sum",fill_value=0,observed=True))
# Of the formerly unmatched cells, which markers actually cleared the required positive gates?
anchor = pd.read_parquet(data / "pln_scimap_no_match_root_markers.parquet")
observed = anchor.groupby(["scimap_candidate_root","rule_term","marker"],observed=True)[["measured","positive"]].sum()
observed["positive_pct"] = 100*observed.positive / observed.measured.replace(0,np.nan)
display(observed.round(2))


In [ ]:
pln_donor = pln[pln.obs.donor.astype(str).eq(DONOR) & np.isfinite(pln.obsm["X_umap"]).all(axis=1)].copy()
fig, axes = plt.subplots(1,2,figsize=(12.0,5.2),sharex=True,sharey=True)
lineage_palette = phenotype_colors(counts, "pLN")
lineage_palette.update({"Unknown":"#777777", "Unassessable":"#c0c0c0"})
for ax,key,title in [(axes[0],"primary","Strict rules"),(axes[1],"scimap_candidate_root","Continuous scoring")]:
    sc.pl.umap(pln_donor, color=key, palette={l:lineage_palette[l] for l in pln_donor.obs[key].cat.categories}, size=11, ax=ax, title=title,
               legend_loc="none", frameon=False, show=False)
    labels = pln_donor.obs[key].cat.categories
    colors_here = dict(zip(labels,pln_donor.uns[key+"_colors"]))
    handles = [Line2D([],[],marker="o",ms=4,ls="",color=colors_here[l],label=subtype_display_name(l)) for l in labels]
    ax.legend(handles=handles,loc="upper center",bbox_to_anchor=(.5,-.02),ncol=2,frameon=False,fontsize=8)
fig.suptitle(f"Donor {DONOR} · pLN broad lineage candidates on the same protein UMAP")
fig.subplots_adjust(bottom=.27)
plt.show()

## 3. Visualize the independent tissue embeddings

Each panel shows the most specific recorded assignment or candidate for the selected donor in the shared coordinate system for that tissue. Distances between panels have no biological interpretation. Small populations are enriched in the display sample; do not count dots to estimate abundance.

In [ ]:
def view(a, donor, tissue):
    keep = a.obs.donor.astype(str).eq(donor).to_numpy() & np.isfinite(a.obsm["X_umap"]).all(axis=1)
    mapped = a[keep].copy()
    mapped.obs["subtype"] = (mapped.obs.scimap_candidate_detail.cat.remove_unused_categories()
                             if tissue == "pLN" and PLN_METHOD == "scimap" else cell_subtypes(mapped.obs, tissue))
    if tissue == "pLN" and PLN_METHOD == "review":
        mapped.obs["subtype"] = mapped.obs.pln_macrophage_detail.cat.remove_unused_categories()
    return mapped
sections = [("Pancreas",view(pan,DONOR,"Pancreas")), ("pLN",view(pln,DONOR,"pLN"))]
colors = {t:phenotype_colors(counts,t, pln_counts.label.unique() if t == "pLN" else ()) for t,_ in sections}
legend_height = max(.23 * np.ceil(a.obs.subtype.nunique()/2) + .25 for _,a in sections)
fig = plt.figure(figsize=(14.5,5.2+legend_height), layout="constrained")
grid = fig.add_gridspec(2,2,height_ratios=[4.5,legend_height])
for col,(t,a) in enumerate(sections):
    ax = fig.add_subplot(grid[0,col])
    sc.pl.umap(a,color="subtype",ax=ax,palette=colors[t],size=12,frameon=False,
               legend_loc="none",show=False,title=f"{t}{' · '+method_name if t == 'pLN' else ''}  ·  donor {DONOR}  ·  {a.n_obs:,} mapped cells")
    phenotype_key(fig.add_subplot(grid[1,col]), names_in_order(a.obs.subtype.dropna().unique(),t),
                  colors[t], columns=2)
plt.show()

## 4. Compare one protein in its tissue context

Pancreas uses the cohort aligned floor/span typing layer. pLN uses its own section-specific clipped scale. Each color bar refers to values in its own tissue; do not interpret the color difference as a fold-change. Missing measurements are shown in grey.

In [ ]:
fig, axs = plt.subplots(1,2,figsize=(10.5,4.2))
for (t,a),ax in zip(sections,axs):
    layer = "typing_scaled" if t=="Pancreas" else None
    m = np.asarray((a.layers[layer] if layer else a.X)[:,a.var_names.get_loc(MARKER)]).ravel()
    if np.isfinite(m).any():
        sc.pl.umap(a,color=MARKER,layer=layer,use_raw=False,ax=ax,cmap="viridis",na_color="#c6cace",
                   size=14,vmin=0,vmax=1,frameon=False,show=False,
                   title=f"{t}  ·  {MARKER}  ·  {np.isfinite(m).sum():,} measured")
    else:
        ax.text(.5,.5,"Marker unavailable",ha="center",va="center",transform=ax.transAxes);ax.set_axis_off()
fig.tight_layout()
plt.show()

## 5. Compare cell composition within each tissue

The pLN panel reports provisional rule outcomes. Unknown, overlapping and unavailable assignments remain visible for the selected method. Bars use every admitted cell of the selected donor in that tissue. Shared words such as “Immune” do not imply matched gates or identical parent populations.

In [ ]:
selected_counts = pd.concat([counts.loc[counts.tissue.eq("Pancreas")], pln_counts], ignore_index=True)
present = selected_counts.loc[selected_counts.donor.eq(DONOR) & selected_counts.level.eq("subtype")].copy()
height = max(5, .26 * present.loc[present.cells.gt(0)].groupby("tissue",observed=True).size().max() + 1)
fig, axes = plt.subplots(1,2,figsize=(14.5,height),sharex=True,layout="constrained")
for ax,t in zip(axes,["Pancreas","pLN"]):
    z = present.loc[present.tissue.eq(t)]
    denom = int(z.eligible_cells.iloc[0])
    assert int(z.cells.sum()) == denom
    z = z.loc[z.cells.gt(0)].sort_values("cells")
    share = 100*z.cells/denom
    ax.barh([subtype_display_name(label) for label in z.label],share,color=[colors[t][v] for v in z.label],height=.66)
    ax.set(xlim=(0,105),xlabel="Admitted cells in this tissue (%)",title=f"{t}{' · '+method_name if t == 'pLN' else ''}  ·  {denom:,} cells")
    ax.tick_params(axis="y", labelsize=8)
    ax.bar_label(ax.containers[0],labels=[f"{v:.2g}%" for v in share],padding=3,size=8)
plt.show()

## 6. Full-panel protein signatures by tissue

Each tissue has its own subtype hierarchy and protein panel. Dotplots show every registered subtype and calibrated typing-panel protein in that tissue, including neural, vascular and lymphatic markers. Dot color scales each marker across subtypes within the displayed tissue and scope. Dot area is the measured fraction ≥0.10. The heatmap retains tissue-specific calibrated means. Gray marks unavailable measurements, including subtypes with no assigned cells.

In [ ]:
pan_markers = show_profiles(counts, evidence, tissue="Pancreas", level="subtype", donor=DONOR)
selection = ({"donor":DONOR} if PLN_PROFILE_SCOPE == "donor"
             else {"disease":str(audit.set_index("donor").at[DONOR,"disease"])} if PLN_PROFILE_SCOPE == "disease"
             else {})
print(f"pLN · {method_name} · {PLN_PROFILE_SCOPE}")
tissue_markers = show_profiles(pln_counts, pln_evidence, tissue="pLN", level="subtype", **selection)
display(tissue_markers.reset_index())